# MLP Classification — Hyperparameter Grid Search

自動搜尋最佳超參數組合（5種模型架構 × learning_rate × weight_decay × dropout_rate）

每個組合跑 **300 epochs**，最後輸出 **Test Macro F1-score 最高**的組合。

In [ ]:
!git clone https://github.com/Anson-ntuim/DL-Final.git


Cloning into 'DL-Final'...


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import re
import itertools
from datetime import datetime, timezone
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## 全域設定與超參數網格

In [ ]:
# ── 固定設定 ─────────────────────────────────
BATCH_SIZE   = 64
NUM_EPOCHS   = 100
TRAIN_RATIO  = 0.8
SEED = 42
NUM_CLASSES  = 4

# 分類邊界定義
CLASS_BOUNDARIES = [10_000, 100_000, 300_000]
CLASS_NAMES = ['Low (<10K)', 'Medium (10K~100K)', 'High (100K~300K)', 'Viral (>=300K)']

# ── 超參數搜尋空間 ────────────────────────────
LEARNING_RATES = [0.001, 0.0005, 0.0001]
WEIGHT_DECAYS  = [1e-3, 1e-4, 1e-5]
DROPOUT_RATES  = [0.1, 0.2, 0.3]

# ── 5 種模型架構 (hidden_dims 列表) ──────────
MODEL_CONFIGS = {
    'Model_A': [512, 256, 128, 64],
    'Model_B': [256, 128, 64],
    'Model_C': [1024, 512, 256, 128, 64],
    'Model_D': [512, 512, 256, 128],
    'Model_E': [256, 256, 128, 128, 64],
}

total_combos = len(MODEL_CONFIGS) * len(LEARNING_RATES) * len(WEIGHT_DECAYS) * len(DROPOUT_RATES)
print(f'總共搜尋組合數: {total_combos} 個（每個跑 {NUM_EPOCHS} epochs）')


總共搜尋組合數: 135 個（每個跑 100 epochs）


## 工具函式 & 資料前處理

In [ ]:
def parse_iso8601_duration(duration_str):
    if not isinstance(duration_str, str):
        return 0
    pattern = r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?'
    match = re.match(pattern, duration_str)
    if not match:
        return 0
    h = int(match.group(1) or 0)
    m = int(match.group(2) or 0)
    s = int(match.group(3) or 0)
    return h * 3600 + m * 60 + s

def parse_published_at(dt_str):
    try:
        dt = datetime.fromisoformat(dt_str.replace('Z', '+00:00'))
        return dt.hour, dt.weekday()
    except Exception:
        return 0, 0

def view_count_to_label(view_count):
    if view_count < 10_000:
        return 0
    elif view_count < 100_000:
        return 1
    elif view_count < 300_000:
        return 2
    else:
        return 3

class StandardScaler:
    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.std_  = X.std(axis=0) + 1e-8
        return self
    def transform(self, X):
        return (X - self.mean_) / self.std_

class YouTubeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [ ]:
# ── 讀取資料 ─────────────────────────────────
data_dir = 'DL-Final/data'
files = os.listdir(data_dir)
csv_files  = [f for f in files if f.endswith('.csv')]
json_files = [f for f in files if f.endswith('.json')]

if csv_files:
    df = pd.read_csv(os.path.join(data_dir, csv_files[0]))
elif json_files:
    df = pd.read_json(os.path.join(data_dir, json_files[0]))
else:
    raise FileNotFoundError('找不到資料檔')

print(f'Loaded data shape={df.shape}')

# ── 特徵工程 ─────────────────────────────────
if 'category_id' in df.columns:
    df = df[df['category_id'] != 10].copy()

df['category_id'] = df['category_id'].astype(str)
df_encoded = pd.get_dummies(df, columns=['category_id'], prefix='cat')

if 'duration_iso8601' in df.columns:
    df_encoded['video_duration_sec'] = df['duration_iso8601'].apply(parse_iso8601_duration)

if 'published_at' in df.columns:
    df_encoded[['pub_hour', 'pub_weekday']] = df['published_at'].apply(
        lambda x: pd.Series(parse_published_at(str(x)))
    )
    df_encoded['pub_hour_sin']    = np.sin(2 * np.pi * df_encoded['pub_hour'] / 24.0)
    df_encoded['pub_hour_cos']    = np.cos(2 * np.pi * df_encoded['pub_hour'] / 24.0)
    df_encoded['pub_weekday_sin'] = np.sin(2 * np.pi * df_encoded['pub_weekday'] / 7.0)
    df_encoded['pub_weekday_cos'] = np.cos(2 * np.pi * df_encoded['pub_weekday'] / 7.0)

df_encoded['subscriber_count']   = np.log1p(df_encoded['subscriber_count'])
df_encoded['video_duration_sec'] = np.log1p(df_encoded['video_duration_sec'])
df_encoded['description_length'] = np.log1p(df_encoded['description_length'])
df_encoded['channel_view_count']  = np.log1p(df_encoded['channel_view_count'])
df_encoded['channel_video_count'] = np.log1p(df_encoded['channel_video_count'])

df['days_since_published'] = df['published_at'].apply(
    lambda x: (datetime.now(timezone.utc) -
               datetime.fromisoformat(str(x).replace('Z', '+00:00'))).days
    if 'T' in str(x) else 0
)
df_encoded['days_since_published'] = np.log1p(df['days_since_published'])

CAT_COLS     = [col for col in df_encoded.columns if col.startswith('cat_')]
NUM_COLS     = ['subscriber_count', 'video_duration_sec', 'tags_count', 'description_length',
                'pub_hour_sin', 'pub_hour_cos', 'pub_weekday_sin', 'pub_weekday_cos',
                'days_since_published', 'channel_view_count', 'channel_video_count']
FEATURE_COLS = CAT_COLS + NUM_COLS

X = df_encoded[FEATURE_COLS].values.astype(np.float32)
INPUT_DIM = X.shape[1]

view_col = 'view_count' if 'view_count' in df.columns else 'viewCount'
y = np.array([view_count_to_label(v) for v in df[view_col].values], dtype=np.int64)

print(f'Feature matrix shape: {X.shape}')
print('Label distribution:')
for i, name in enumerate(CLASS_NAMES):
    count = (y == i).sum()
    print(f'  Class {i} [{name}]: {count} samples ({count/len(y)*100:.1f}%)')


Loaded data shape=(34383, 27)
Feature matrix shape: (34383, 25)
Label distribution:
  Class 0 [Low (<10K)]: 28188 samples (82.0%)
  Class 1 [Medium (10K~100K)]: 4936 samples (14.4%)
  Class 2 [High (100K~300K)]: 893 samples (2.6%)
  Class 3 [Viral (>=300K)]: 366 samples (1.1%)


In [ ]:
# ── 固定 Train/Test split ────────────────────
np.random.seed(SEED)
torch.manual_seed(SEED)

n_total = len(X)
n_train = int(n_total * TRAIN_RATIO)
indices = np.random.permutation(n_total)
train_idx, test_idx = indices[:n_train], indices[n_train:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

scaler  = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test  = scaler.transform(X_test)

train_dataset = YouTubeDataset(X_train, y_train)
test_dataset  = YouTubeDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train: {len(train_idx)} | Test: {len(test_idx)}')


Train: 27506 | Test: 6877


## 模型定義

In [ ]:
class myActivation(nn.Module):
    def __init__(self):
        super().__init__()
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(x)


class myLoss(nn.Module):
    """Cross Entropy Loss for multi-class classification"""
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
    def forward(self, pred, target):
        return self.ce(pred, target)


class FlexMLP(nn.Module):
    """
    可彈性設定層數與寬度的 MLP（分類版）。
    hidden_dims: list of int，例如 [512, 256, 128, 64]
    """
    def __init__(self, input_dim, hidden_dims, dropout_rate=0.1, num_classes=4):
        super().__init__()
        layers = []
        in_dim = input_dim
        for i, h_dim in enumerate(hidden_dims):
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(myActivation())
            if i < len(hidden_dims) - 1:
                layers.append(nn.Dropout(dropout_rate))
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, num_classes))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        return self.mlp(x)


print('模型類別定義完成')


模型類別定義完成


## 訓練函式

In [ ]:
def train_and_evaluate(model_name, hidden_dims, lr, wd, dr, num_epochs=NUM_EPOCHS):
    torch.manual_seed(SEED)
    model = FlexMLP(INPUT_DIM, hidden_dims, dropout_rate=dr, num_classes=NUM_CLASSES).to(device)
    criterion = myLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=15
    )

    for epoch in range(num_epochs):
        model.train()
        losses = []
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device).float()
            y_batch = y_batch.to(device).long()
            optimizer.zero_grad()
            output = model(x_batch)
            loss   = criterion(output, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            losses.append(loss.item())
        if epoch % 10 == 0:
          avg_loss = np.mean(losses)
          scheduler.step(avg_loss)
          print(f'    Epoch {epoch+1:3d}/{num_epochs} | Loss: {avg_loss:.4f}')

    # ── Evaluation ──
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            logits = model(x_batch.to(device).float())
            preds  = torch.argmax(logits, dim=1)
            all_preds.append(preds.cpu())
            all_labels.append(y_batch.cpu())

    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    overall_acc = (all_preds == all_labels).mean()
    macro_f1    = f1_score(all_labels, all_preds, average='macro')
    return overall_acc, macro_f1


print('訓練函式定義完成')


訓練函式定義完成


## 超參數搜尋（Grid Search）

> ⏳ 共 135 組合 × 300 epochs，請耐心等待。有 GPU 會快很多。

In [ ]:
results = []

all_combos = list(itertools.product(
    MODEL_CONFIGS.items(),
    LEARNING_RATES,
    WEIGHT_DECAYS,
    DROPOUT_RATES
))

total = len(all_combos)
best_f1     = -1
best_config = None

SEP  = '─' * 62
SEP2 = '═' * 62

print(SEP2)
print(f'  🔍 超參數 Grid Search 開始  （共 {total} 個組合）')
print(SEP2)
print()

for i, ((model_name, hidden_dims), lr, wd, dr) in enumerate(all_combos):

    print(SEP)
    print(f'  [{i+1:3d}/{total}]  正在訓練...')
    print(f'  模型架構  : {model_name}  {hidden_dims}')
    print(f'  lr        : {lr}    wd : {wd:.0e}    dropout : {dr}')
    print(SEP)

    acc, macro_f1 = train_and_evaluate(model_name, hidden_dims, lr, wd, dr)

    is_new_best = macro_f1 > best_f1
    if is_new_best:
        best_f1     = macro_f1
        best_config = {
            'model_name':    model_name,
            'hidden_dims':   str(hidden_dims),
            'learning_rate': lr,
            'weight_decay':  wd,
            'dropout_rate':  dr,
            'test_acc':      acc,
            'macro_f1':      macro_f1
        }

    results.append({
        'model_name':    model_name,
        'hidden_dims':   str(hidden_dims),
        'learning_rate': lr,
        'weight_decay':  wd,
        'dropout_rate':  dr,
        'test_acc':      acc,
        'macro_f1':      macro_f1
    })

    star = '  ★ NEW BEST!' if is_new_best else ''
    print(f'  結果  →  Accuracy: {acc*100:.2f}%  |  Macro F1: {macro_f1:.4f}{star}')
    print(f'  目前最佳 Macro F1 : {best_f1:.4f}'
          f'  ({best_config["model_name"]}'
          f'  lr={best_config["learning_rate"]}'
          f'  wd={best_config["weight_decay"]:.0e}'
          f'  dr={best_config["dropout_rate"]})')
    print()

print(SEP2)
print('  ✅  搜尋完畢！')
print(SEP2)
print()
print('  🏆  最佳組合')
print(f'  模型架構     : {best_config["model_name"]}  {best_config["hidden_dims"]}')
print(f'  learning_rate: {best_config["learning_rate"]}')
print(f'  weight_decay : {best_config["weight_decay"]}')
print(f'  dropout_rate : {best_config["dropout_rate"]}')
print(f'  Test Accuracy: {best_config["test_acc"]*100:.2f}%')
print(f'  Macro F1     : {best_config["macro_f1"]:.4f}')
print(SEP2)


## 結果分析

In [ ]:
df_results = pd.DataFrame(results).sort_values('macro_f1', ascending=False).reset_index(drop=True)

print('=' * 75)
print('Top 10 最佳組合（按 Macro F1 排序）')
print('=' * 75)
print(df_results.head(10).to_string(index=False))

df_results.to_csv('cls_hyperparam_search_results.csv', index=False)
print('\n結果已儲存至 cls_hyperparam_search_results.csv')


## 視覺化結果

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

model_avg = df_results.groupby('model_name')['macro_f1'].max().sort_values()
axes[0].barh(model_avg.index, model_avg.values, color='steelblue')
axes[0].set_xlabel('Best Macro F1')
axes[0].set_title('Best F1 by Model Architecture')

lr_avg = df_results.groupby('learning_rate')['macro_f1'].max()
axes[1].bar([str(lr) for lr in lr_avg.index], lr_avg.values, color='coral')
axes[1].set_xlabel('Learning Rate')
axes[1].set_title('Best F1 by Learning Rate')

dr_avg = df_results.groupby('dropout_rate')['macro_f1'].max()
axes[2].bar([str(dr) for dr in dr_avg.index], dr_avg.values, color='mediumseagreen')
axes[2].set_xlabel('Dropout Rate')
axes[2].set_title('Best F1 by Dropout Rate')

plt.suptitle('Classification Hyperparameter Search Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cls_hyperparam_search_plot.png', dpi=120, bbox_inches='tight')
plt.show()
print('圖表已儲存至 cls_hyperparam_search_plot.png')


## 用最佳組合重新訓練最終模型

In [ ]:
best_row        = df_results.iloc[0]
BEST_MODEL_NAME = best_row['model_name']
BEST_HIDDEN     = MODEL_CONFIGS[BEST_MODEL_NAME]
BEST_LR         = best_row['learning_rate']
BEST_WD         = best_row['weight_decay']
BEST_DR         = best_row['dropout_rate']

print(f'以最佳組合重新訓練 {NUM_EPOCHS} epochs...')
print(f'  {BEST_MODEL_NAME} {BEST_HIDDEN} | lr={BEST_LR} wd={BEST_WD} dr={BEST_DR}')

torch.manual_seed(SEED)
final_model = FlexMLP(INPUT_DIM, BEST_HIDDEN, dropout_rate=BEST_DR, num_classes=NUM_CLASSES).to(device)
criterion   = myLoss()
optimizer   = optim.Adam(final_model.parameters(), lr=BEST_LR, weight_decay=BEST_WD)
scheduler   = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=15
)

for epoch in range(NUM_EPOCHS):
    final_model.train()
    losses = []
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device).float()
        y_batch = y_batch.to(device).long()
        optimizer.zero_grad()
        out  = final_model(x_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(final_model.parameters(), max_norm=1.0)
        optimizer.step()
        losses.append(loss.item())
    avg_loss = np.mean(losses)
    scheduler.step(avg_loss)
    print(f'    Epoch {epoch+1:3d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f}')

# ── Final evaluation ──
final_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x_batch, y_batch in test_loader:
        logits = final_model(x_batch.to(device).float())
        preds  = torch.argmax(logits, dim=1)
        all_preds.append(preds.cpu())
        all_labels.append(y_batch.cpu())

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

final_acc  = (all_preds == all_labels).mean()
final_f1   = f1_score(all_labels, all_preds, average='macro')

print('\n' + '=' * 55)
print('最終模型評估結果（最佳超參數）')
print('=' * 55)
print(f'Overall Accuracy : {final_acc*100:.2f}%')
print(f'Macro F1-score   : {final_f1:.4f}')
print()
print('Per-class Accuracy:')
for i, name in enumerate(CLASS_NAMES):
    mask = (all_labels == i)
    if mask.sum() == 0:
        print(f'  Class {i} [{name}]: N/A')
    else:
        cls_acc = (all_preds[mask] == all_labels[mask]).mean()
        print(f'  Class {i} [{name}]: {cls_acc*100:.2f}% ({mask.sum()} samples)')
print('=' * 55)

torch.save(final_model.state_dict(), 'best_cls_model_state_dict.pt')
print('模型已儲存至 best_cls_model_state_dict.pt')


In [ ]:
import matplotlib.pyplot as plt

conf_matrix = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
for t, p in zip(all_labels, all_preds):
    conf_matrix[t][p] += 1

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 左圖：Confusion Matrix
im = axes[0].imshow(conf_matrix, cmap='Blues')
axes[0].set_xticks(range(NUM_CLASSES))
axes[0].set_yticks(range(NUM_CLASSES))
axes[0].set_xticklabels([f'Pred {i}\n{n}' for i, n in enumerate(CLASS_NAMES)], fontsize=8)
axes[0].set_yticklabels([f'True {i}\n{n}' for i, n in enumerate(CLASS_NAMES)], fontsize=8)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        axes[0].text(j, i, str(conf_matrix[i][j]),
                     ha='center', va='center',
                     color='white' if conf_matrix[i][j] > conf_matrix.max()/2 else 'black')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')
axes[0].set_title(f'Confusion Matrix (Best Model)\n(Accuracy: {final_acc*100:.2f}%, Macro F1: {final_f1:.4f})')
plt.colorbar(im, ax=axes[0])

# 右圖：Class distribution
x_pos = np.arange(NUM_CLASSES)
true_counts = [(all_labels == i).sum() for i in range(NUM_CLASSES)]
pred_counts = [(all_preds  == i).sum() for i in range(NUM_CLASSES)]

width = 0.35
axes[1].bar(x_pos - width/2, true_counts, width, label='Ground Truth', color='steelblue', alpha=0.8)
axes[1].bar(x_pos + width/2, pred_counts, width, label='Predicted',    color='coral',     alpha=0.8)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'Class {i}\n{n}' for i, n in enumerate(CLASS_NAMES)], fontsize=8)
axes[1].set_ylabel('Count')
axes[1].set_title('Class Distribution: Ground Truth vs Predicted')
axes[1].legend()

plt.suptitle(
    f'Best: {BEST_MODEL_NAME} | lr={BEST_LR} wd={BEST_WD:.0e} dr={BEST_DR}',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('best_cls_model_result.png', dpi=120, bbox_inches='tight')
plt.show()
print('圖表已儲存至 best_cls_model_result.png')
